<a href="https://colab.research.google.com/github/PYMaksim/llm_prompt_engineering_experiments/blob/main/llm_prompt_engineering_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# БЛОК 1: Установка и настройка
# BLOCK 1: Setup & Configuration
# ============================================================

# --- Шаг 1: Установка библиотек / Step 1: Install libraries ---
# openai — официальный клиент OpenAI API (если есть ключ)
# openai — official OpenAI API client (if you have a key)
# transformers + torch — для бесплатного запуска модели через Hugging Face
# transformers + torch — for free local model via Hugging Face
# ---------------------------------------------------------
# Устанавливаем тихо, без лишнего вывода (флаг -q)
# Install quietly, suppressing extra output (-q flag)

print("Устанавливаю библиотеки / Installing libraries...")

!pip install -q openai
!pip install -q transformers torch

print("Готово / Done.")

# --- Шаг 2: Импорты / Step 2: Imports ---
# json — для работы с JSON-ответами (валидация, парсинг)
# json — for working with JSON responses (validation, parsing)
# textwrap — для красивого вывода длинного текста в консоль
# textwrap — for pretty-printing long text in console
# ---------------------------------------------------------

import json
import textwrap

print("Импорты выполнены / Imports done.")

# --- Шаг 3: Конфигурация бэкенда / Step 3: Backend configuration ---
# BACKEND = "openai"      → если у тебя есть API-ключ OpenAI
# BACKEND = "openai"      → if you have an OpenAI API key
# BACKEND = "huggingface" → бесплатный вариант (нужен GPU в Colab)
# BACKEND = "huggingface" → free option (requires GPU in Colab)
# ---------------------------------------------------------
# Для бесплатного варианта:
# For the free option:
#   Runtime → Change runtime type → T4 GPU
# Модель: Qwen2.5-1.5B-Instruct — маленькая, но для наших задач хватает
# Model: Qwen2.5-1.5B-Instruct — small, but sufficient for our tasks
# ---------------------------------------------------------

BACKEND = "huggingface"  # поменяй на "openai" если есть ключ / change to "openai" if you have a key

# Настройки для OpenAI (если используешь) / Settings for OpenAI (if used)
OPENAI_MODEL = "gpt-4o-mini"

# Настройки для Hugging Face (если используешь) / Settings for Hugging Face (if used)
HF_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"Бэкенд / Backend: {BACKEND}")
print(f"Модель / Model:   {OPENAI_MODEL if BACKEND == 'openai' else HF_MODEL}")
print("=" * 50)
print("Блок 1 готов. Запусти эту ячейку первой.")
print("Block 1 is ready. Run this cell first.")


Устанавливаю библиотеки / Installing libraries...
Готово / Done.
Импорты выполнены / Imports done.
Бэкенд / Backend: huggingface
Модель / Model:   Qwen/Qwen2.5-1.5B-Instruct
Блок 1 готов. Запусти эту ячейку первой.
Block 1 is ready. Run this cell first.


In [4]:
# ============================================================
# БЛОК 2: Загрузка модели и функция вызова (ИСПРАВЛЕННАЯ ВЕРСИЯ)
# BLOCK 2: Model loading and call function (FIXED VERSION)
# ============================================================

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# --- Шаг 1: Инициализация токенизатора и модели / Step 1: Initialize tokenizer and model ---
# AutoTokenizer — превращает текст в числа (токены), понятные модели
# AutoTokenizer — converts text into numbers (tokens) the model understands
# AutoModelForCausalLM — сама языковая модель, которая генерирует продолжение текста
# AutoModelForCausalLM — the language model that generates text continuation
# ---------------------------------------------------------

print("Загружаю модель... / Loading model...")

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL)

model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL,
    torch_dtype=torch.float16,      # float16 saves memory and speeds up on GPU
    device_map="auto"               # автоматически размещает модель на доступном устройстве (GPU/CPU)
    # device_map="auto"             # automatically places model on available device (GPU/CPU)
)



model = AutoModelForCausalLM.from_pretrained(
    HF_MODEL,
    torch_dtype=torch.float16,       # float16 для экономии памяти и скорости
    device_map="auto"                # авто-распределение по устройствам (GPU/CPU)
)

print("Модель загружена. / Model loaded.")

# --- Шаг 2: Функция вызова модели / Step 2: Model call function ---
# Эта функция принимает промпт и возвращает ответ модели
# This function takes a prompt and returns the model's response
# ---------------------------------------------------------

def call_hf_model(prompt: str) -> str:
    # Токенизируем промпт: превращаем текст в массив чисел
    # Tokenize the prompt: convert text to array of numbers
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Генерируем ответ
    # Generate the response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,          # максимум 512 новых токенов (примерно 300–400 слов)
            temperature=0.7,             # температура 0.7 — баланс между креативностью и стабильностью
            do_sample=True,              # разрешаем выборку (не только самый вероятный токен)
            pad_token_id=tokenizer.eos_token_id  # чтобы не было ошибок с паддингом
        )

    # Превращаем числа обратно в текст
    # Convert numbers back to text
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Возвращаем только ответ модели (убираем исходный промпт из вывода)
    # Return only the model's answer (strip the original prompt from output)
    return response[len(prompt):].strip()

print("Функция call_hf_model готова. / Function call_hf_model is ready.")
print("=" * 50)
print("Блок 2 готов. Запусти эту ячейку второй.")
print("Block 2 is ready. Run this cell second.")


Загружаю модель... / Loading model...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Модель загружена. / Model loaded.
Функция call_hf_model готова. / Function call_hf_model is ready.
Блок 2 готов. Запусти эту ячейку второй.
Block 2 is ready. Run this cell second.


In [5]:
# ============================================================
# БЛОК 3: Промпты (RU + EN), вызов и сравнение результатов
# BLOCK 3: Prompts (RU + EN), call, and results comparison
# ============================================================

from textwrap import shorten

# --- Шаг 1: Тестовый контекст (RU и EN) / Step 1: Test context (RU & EN) ---
# Один и тот же смысл на двух языках — для честного сравнения
# The same meaning in two languages — for fair comparison
# ---------------------------------------------------------

context_ru = """
Проект внедряет чат‑бота для поддержки клиентов. В базе знаний 1000 статей. Бот иногда отвечает выдуманными фактами. Есть риск, что пользователи получат неверные инструкции по безопасности. Команда хочет снизить количество галлюцинаций и контролировать тон ответов.
"""

context_en = """
The project is deploying a chatbot for customer support. The knowledge base contains 1,000 articles. The bot sometimes answers with made‑up facts. There is a risk that users will receive incorrect safety instructions. The team wants to reduce hallucinations and control the tone of responses.
"""

print("Контекст готов (RU/EN). / Context is ready (RU/EN).")

# --- Шаг 2: Три структуры промптов (RU и EN версии) / Step 2: Three prompt structures (RU & EN versions) ---

# Промпт A: Zero-shot (минимальные инструкции) / Prompt A: Zero-shot (minimal instructions)
prompt_zero_shot_ru = f"""
Роль: аналитик рисков.
Задача: выпиши из текста риски и меры по ним.
Формат: простой нумерованный список.
Ограничение: не выдумывай факты, используй только предоставленный текст. Если информации нет — напиши «недостаточно данных».

Контекст:
{context_ru}
"""

prompt_zero_shot_en = f"""
Role: risk analyst.
Task: list the risks and corresponding mitigation measures from the text.
Format: simple numbered list.
Constraint: do not invent facts; use only the provided text. If there is no information, write “insufficient data”.

Context:
{context_en}
"""

# Промпт B: Few-shot (с примерами) / Prompt B: Few-shot (with examples)
prompt_few_shot_ru = f"""
Роль: аналитик рисков.

Примеры (вход → выход):
Текст: «Бот иногда ошибается в датах».
Риск: неверная информация о сроках.
Мера: добавить проверку дат по календарю.

Текст: «Нет контроля тона ответов».
Риск: токсичные ответы.
Мера: внедрить фильтр тона и правила стиля.

Задача: выпиши из предоставленного текста риски и меры по ним по этому же шаблону.
Формат: список пар «Риск: ... / Мера: ...».
Ограничение: не выдумывай факты. Если информации нет — напиши «недостаточно данных».

Контекст:
{context_ru}
"""

prompt_few_shot_en = f"""
Role: risk analyst.

Examples (input → output):
Text: “The bot sometimes makes mistakes with dates.”
Risk: incorrect information about deadlines.
Mitigation: add date validation against a calendar.

Text: “There is no tone control for responses.”
Risk: toxic responses.
Mitigation: implement a tone filter and style guidelines.

Task: list risks and mitigation measures from the provided text using the same pattern.
Format: list of pairs “Risk: … / Mitigation: …”.
Constraint: do not invent facts. If there is no information, write “insufficient data”.

Context:
{context_en}
"""

# Промпт C: Chain-of-Thought + строгий JSON + этика / Prompt C: Chain-of-Thought + strict JSON + ethics
prompt_cot_ru = f"""
Роль: аналитик рисков с фокусом на безопасность и этику ИИ.

Шаги:
1. Сначала выпиши все потенциальные риски из текста (только факты из текста).
2. Для каждого риска кратко оцени серьёзность (низкая/средняя/высокая) и обоснуй одной фразой.
3. Предложи меры, которые реально можно внедрить в ближайший спринт.
4. Выведи ТОЛЬКО JSON со следующей схемой:
{{
  "risks": [
    {{
      "id": "R1",
      "description": "...",
      "severity": "низкая/средняя/высокая",
      "justification": "...",
      "mitigation": "..."
    }}
  ]
}}
Ограничения:
- Не выдумывай факты.
- Не используй дискриминационные или токсичные формулировки.
- Если данных недостаточно для какого‑то поля — оставь null.
- Никаких пояснений вне JSON.

Контекст:
{context_ru}
"""

prompt_cot_en = f"""
Role: risk analyst with a focus on AI safety and ethics.

Steps:
1. First, list all potential risks from the text (only facts from the text).
2. For each risk, briefly assess severity (low/medium/high) and justify in one sentence.
3. Propose measures that can realistically be implemented in the next sprint.
4. Output ONLY JSON with the following schema:
{{
  "risks": [
    {{
      "id": "R1",
      "description": "...",
      "severity": "low/medium/high",
      "justification": "...",
      "mitigation": "..."
    }}
  ]
}}
Constraints:
- Do not invent facts.
- Do not use discriminatory or toxic language.
- If there is insufficient data for a field, leave it as null.
- No explanations outside the JSON.

Context:
{context_en}
"""

print("Промпты сформированы (RU/EN). / Prompts are ready (RU/EN).")

# --- Шаг 3: Вызов модели и вывод результатов / Step 3: Call model and display results ---

def run_and_print(label_ru, label_en, prompt_ru, prompt_en):
    print("\n" + "=" * 60)
    print(f"Тест: {label_ru} / {label_en}")
    print("=" * 60)

    # Русский ответ
    print("--- Ответ на русском / Russian response ---")
    resp_ru = call_hf_model(prompt_ru)
    if len(resp_ru) > 800:
        print(shorten(resp_ru, width=800, placeholder="... [текст обрезан]"))
    else:
        print(resp_ru)

    # Английский ответ
    print("\n--- Ответ на английском / English response ---")
    resp_en = call_hf_model(prompt_en)
    if len(resp_en) > 800:
        print(shorten(resp_en, width=800, placeholder="... [text truncated]"))
    else:
        print(resp_en)

# Запускаем все 3 пары промптов
run_and_print("Zero-shot", "Zero-shot", prompt_zero_shot_ru, prompt_zero_shot_en)
run_and_print("Few-shot", "Few-shot", prompt_few_shot_ru, prompt_few_shot_en)
run_and_print("CoT + JSON", "CoT + JSON", prompt_cot_ru, prompt_cot_en)

print("\n" + "=" * 60)
print("Блок 3 выполнен. / Block 3 is done.")



Контекст готов (RU/EN). / Context is ready (RU/EN).
Промпты сформированы (RU/EN). / Prompts are ready (RU/EN).

Тест: Zero-shot / Zero-shot
--- Ответ на русском / Russian response ---
Вопрос: Что рискует проект внедрения чат-бота? Какие меры можно предпринять для минимизации этих рисков? Ответ: 1. Неправильная информация может привести к проблемам безопасности в отношении пользователей. 2. **Меры**: - Использовать проверенные источники для создания статей о боте. - Проводить обучение на основе реальных случаев использования чат-бота. - Обеспечивать регулярное обновление базы данных. 3. Возможно, стоит также создать функцию для обратной связи пользователям, которая поможет им сообщить о возможных проблемах или недоразумениях с чатом. Это позволит операторам лучше понимать потребности клиентов и улучшить качество обслуживания. --- ### Результат выполнения 1. **Неправильная информация может привести к проблемам безопасности в отношении пользователей.**... [текст обрезан]

--- Ответ на анг

In [7]:
# ============================================================
# БЛОК 4: Валидация, таблица сравнения, сохранение артефактов
# BLOCK 4: Validation, comparison table, saving artifacts
# ============================================================

import json
from textwrap import shorten

# --- Функция валидации JSON / JSON validation function ---
# Проверяет, является ли строка валидным JSON и содержит ли поле "risks"
# Checks whether a string is valid JSON and contains the "risks" field
# ---------------------------------------------------------

def validate_json(text: str):
    try:
        data = json.loads(text)
        return True, data
    except json.JSONDecodeError as e:
        return False, str(e)

# --- Перезапуск CoT-промптов для чистых ответов / Re-run CoT prompts for clean responses ---
# Модель генерирует заново — нам нужны свежие ответы для валидации
# Model generates again — we need fresh responses for validation
# ---------------------------------------------------------

print("Перезапуск CoT-запросов / Re-running CoT queries...")

resp_cot_ru = call_hf_model(prompt_cot_ru)
resp_cot_en = call_hf_model(prompt_cot_en)

# --- Валидация JSON / JSON validation ---
# Проверяем оба ответа (RU и EN) на соответствие формату JSON
# Validate both responses (RU & EN) for JSON format compliance
# ---------------------------------------------------------

ok_ru, result_ru = validate_json(resp_cot_ru)
ok_en, result_en = validate_json(resp_cot_en)

print("\n" + "=" * 60)
print("РЕЗУЛЬТАТЫ ВАЛИДАЦИИ JSON / JSON VALIDATION RESULTS")
print("=" * 60)

# Вывод для русского ответа / Output for Russian response
if ok_ru:
    print("✅ RU JSON валиден / RU JSON is valid")
    risks_ru = result_ru.get("risks", [])
    # Показываем первые 2 риска для наглядности / Show first 2 risks for clarity
    for r in risks_ru[:2]:
        print(f"  - {r['id']}: {r['description']} (severity={r['severity']})")
else:
    print(f"❌ RU JSON невалиден / RU JSON is invalid: {result_ru}")

# Вывод для английского ответа / Output for English response
if ok_en:
    print("✅ EN JSON валиден / EN JSON is valid")
    risks_en = result_en.get("risks", [])
    for r in risks_en[:2]:
        print(f"  - {r['id']}: {r['description']} (severity={r['severity']})")
else:
    print(f"❌ EN JSON невалиден / EN JSON is invalid: {result_en}")

# --- Таблица сравнения техник / Comparison table ---
# Сводка по всем техникам и языкам — для README и анализа
# Summary across all techniques and languages — for README and analysis
# ---------------------------------------------------------

print("\n" + "=" * 60)
print("ТАБЛИЦА СРАВНЕНИЯ ТЕХНИК / TECHNIQUE COMPARISON TABLE")
print("=" * 60)

# Данные для таблицы / Table data
comparison_table = [
    # Technique, Language, Format kept?, Parseable JSON?, Notes
    ["Zero-shot", "RU", "No", "No", "Adds intro text («Question:… Answer:…»), breaks automation"],
    ["Zero-shot", "EN", "No", "No", "Extra phrases, not a clean list"],
    ["Few-shot", "RU", "Partial", "No", "Pattern partially held, but adds comments and code blocks"],
    ["Few-shot", "EN", "Partial", "No", "Better pattern adherence, but still adds explanations"],
    ["CoT + JSON", "RU", "Yes", "Yes" if ok_ru else "No", f"JSON valid: {ok_ru}. Severity field needs normalization"],
    ["CoT + JSON", "EN", "Yes", "Yes" if ok_en else "No", f"JSON valid: {ok_en}. Cleaner format than RU"],
]

# Заголовок таблицы / Table header
header = f"{'Technique':<12} {'Lang':<6} {'Format':<8} {'JSON?':<8} {'Notes'}"
print(header)
print("-" * len(header))

# Вывод строк таблицы / Print table rows
for row in comparison_table:
    tech, lang, fmt, js, notes = row
    print(f"{tech:<12} {lang:<6} {fmt:<8} {js:<8} {notes}")

# --- Сохранение артефактов / Saving artifacts ---
# Записываем всё в results.json — чтобы положить в Git
# Write everything to results.json — to commit to Git
# ---------------------------------------------------------

artifacts = {
    "raw_responses": {
        "cot_ru": resp_cot_ru,
        "cot_en": resp_cot_en
    },
    "validated_json": {
        "ru_valid": ok_ru,
        "en_valid": ok_en,
        "ru_data": result_ru if ok_ru else None,
        "en_data": result_en if ok_en else None
    },
    "comparison_table": comparison_table
}

# Запись в файл / Write to file
with open("results.json", "w", encoding="utf-8") as f:
    json.dump(artifacts, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 60)
print("АРТЕФАКТЫ СОХРАНЕНЫ / ARTIFACTS SAVED")
print("=" * 60)
print("Файл results.json создан в среде Colab. / File results.json created in Colab.")



Перезапуск CoT-запросов / Re-running CoT queries...

РЕЗУЛЬТАТЫ ВАЛИДАЦИИ JSON / JSON VALIDATION RESULTS
❌ RU JSON невалиден / RU JSON is invalid: Expecting value: line 1 column 1 (char 0)
❌ EN JSON невалиден / EN JSON is invalid: Expecting value: line 1 column 1 (char 0)

ТАБЛИЦА СРАВНЕНИЯ ТЕХНИК / TECHNIQUE COMPARISON TABLE
Technique    Lang   Format   JSON?    Notes
-------------------------------------------
Zero-shot    RU     No       No       Adds intro text («Question:… Answer:…»), breaks automation
Zero-shot    EN     No       No       Extra phrases, not a clean list
Few-shot     RU     Partial  No       Pattern partially held, but adds comments and code blocks
Few-shot     EN     Partial  No       Better pattern adherence, but still adds explanations
CoT + JSON   RU     Yes      No       JSON valid: False. Severity field needs normalization
CoT + JSON   EN     Yes      No       JSON valid: False. Cleaner format than RU

АРТЕФАКТЫ СОХРАНЕНЫ / ARTIFACTS SAVED
Файл results.json 